# ECG Signal Analysis & ML-Based Disease Prediction 🫀

This notebook demonstrates a complete pipeline for:
1. Generating synthetic ECG signals (Normal, Arrhythmia, Myocardial Infarction, Atrial Fibrillation, Bradycardia)
2. Preprocessing ECG signals (bandpass filtering, normalization, R-peak detection)
3. Extracting time-domain and frequency-domain features
4. Training ML models (Random Forest, Gradient Boosting, SVM) to predict cardiac conditions
5. Evaluating model performance

> **Note:** This demo uses synthetic ECG data. For production use, replace with real ECG data such as the MIT-BIH Arrhythmia Database.

## 1. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from ecg_analysis.preprocessing import preprocess_ecg
from ecg_analysis.feature_extraction import extract_features
from ecg_analysis.model import ECGClassifier, DISEASE_LABELS
from ecg_analysis.generate_synthetic_data import (
    generate_single_ecg,
    generate_dataset,
    generate_feature_dataset,
)

print('All imports successful!')
print(f'Disease classes: {DISEASE_LABELS}')

## 2. Visualize Synthetic ECG Signals

Let's generate and visualize one ECG signal for each cardiac condition.

In [ ]:
conditions = {
    'Normal': None,
    'Arrhythmia': 'arrhythmia',
    'Myocardial Infarction': 'mi',
    'Atrial Fibrillation': 'afib',
    'Bradycardia': 'bradycardia',
}

fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
fs = 360
duration = 5  # seconds

for ax, (name, abn) in zip(axes, conditions.items()):
    sig = generate_single_ecg(duration=duration, fs=fs, abnormality=abn, random_state=42)
    t = np.linspace(0, duration, len(sig))
    ax.plot(t, sig, linewidth=0.8)
    ax.set_ylabel('Amplitude')
    ax.set_title(name)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('Synthetic ECG Signals by Condition', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Preprocessing Pipeline

Demonstrate bandpass filtering, normalization, and R-peak detection on a normal ECG.

In [ ]:
raw_ecg = generate_single_ecg(duration=5, fs=fs, heart_rate=72, random_state=0)
result = preprocess_ecg(raw_ecg, fs=fs)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
t = np.linspace(0, 5, len(raw_ecg))

axes[0].plot(t, raw_ecg, linewidth=0.8)
axes[0].set_title('Raw ECG Signal')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, result['filtered'], linewidth=0.8, color='green')
axes[1].set_title('Bandpass Filtered')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, result['normalized'], linewidth=0.8, color='blue')
r_peaks = result['r_peaks']
axes[2].plot(t[r_peaks], result['normalized'][r_peaks], 'rv', markersize=8, label='R-peaks')
axes[2].set_title('Normalized with R-Peak Detection')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Amplitude')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Detected {len(r_peaks)} R-peaks in 5 seconds')
print(f'Estimated heart rate: {len(r_peaks) * 12:.0f} bpm')

## 4. Feature Extraction

Extract time-domain and frequency-domain features from the preprocessed signal.

In [ ]:
features = extract_features(result['normalized'], fs=fs, r_peaks=result['r_peaks'])

print('Extracted Features:')
print('=' * 40)
for name, value in features.items():
    print(f'{name:>20s}: {value:.4f}')

## 5. Generate Training Dataset

Create a feature dataset with 50 samples per class (250 total).

In [ ]:
X, y = generate_feature_dataset(n_samples_per_class=50, duration=10, fs=360, random_state=42)

print(f'Feature matrix shape: {X.shape}')
print(f'Label distribution:')
for label_id, name in DISEASE_LABELS.items():
    print(f'  {name}: {np.sum(y == label_id)} samples')

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {len(X_train)} samples')
print(f'Test set:     {len(X_test)} samples')

## 6. Train & Evaluate Models

### 6.1 Random Forest

In [ ]:
rf_clf = ECGClassifier(model_type='random_forest', n_estimators=100, random_state=42)
rf_clf.train(X_train, y_train)
rf_results = rf_clf.evaluate(X_test, y_test)

print(f'Random Forest Accuracy: {rf_results["accuracy"]:.4f}')
print()
print('Classification Report:')
print(rf_results['classification_report'])

### 6.2 Gradient Boosting

In [ ]:
gb_clf = ECGClassifier(model_type='gradient_boosting', n_estimators=100, random_state=42)
gb_clf.train(X_train, y_train)
gb_results = gb_clf.evaluate(X_test, y_test)

print(f'Gradient Boosting Accuracy: {gb_results["accuracy"]:.4f}')
print()
print('Classification Report:')
print(gb_results['classification_report'])

### 6.3 SVM

In [ ]:
svm_clf = ECGClassifier(model_type='svm', random_state=42)
svm_clf.train(X_train, y_train)
svm_results = svm_clf.evaluate(X_test, y_test)

print(f'SVM Accuracy: {svm_results["accuracy"]:.4f}')
print()
print('Classification Report:')
print(svm_results['classification_report'])

### 6.4 Model Comparison

In [ ]:
models = {
    'Random Forest': rf_results['accuracy'],
    'Gradient Boosting': gb_results['accuracy'],
    'SVM': svm_results['accuracy'],
}

plt.figure(figsize=(8, 5))
bars = plt.bar(models.keys(), models.values(), color=['#2ecc71', '#3498db', '#e74c3c'])
plt.ylabel('Accuracy')
plt.title('Model Comparison — ECG Disease Prediction')
plt.ylim(0, 1.0)
for bar, acc in zip(bars, models.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{acc:.2%}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Cross-Validation

In [ ]:
cv_clf = ECGClassifier(model_type='random_forest', n_estimators=100, random_state=42)
cv_results = cv_clf.cross_validate(X, y, cv=5)

print(f'5-Fold Cross-Validation (Random Forest):')
print(f'  Mean Accuracy: {cv_results["mean_accuracy"]:.4f}')
print(f'  Std Accuracy:  {cv_results["std_accuracy"]:.4f}')

## 8. Confusion Matrix Visualization

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=rf_results['confusion_matrix'],
    display_labels=[DISEASE_LABELS[i] for i in sorted(DISEASE_LABELS.keys())]
)
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix — Random Forest')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 9. Predict on New ECG Signals

Demonstrate end-to-end prediction on a new synthetic ECG signal.

In [ ]:
# Generate a new ECG signal with arrhythmia
new_ecg = generate_single_ecg(duration=10, fs=360, abnormality='arrhythmia', random_state=99)

# Preprocess
preprocessed = preprocess_ecg(new_ecg, fs=360)

# Extract features
new_features = extract_features(
    preprocessed['normalized'], fs=360, r_peaks=preprocessed['r_peaks']
)
new_X = pd.DataFrame([new_features])

# Predict
prediction = rf_clf.predict_disease(new_X)
probabilities = rf_clf.predict_proba(new_X)

print(f'Predicted condition: {prediction[0]}')
print()
print('Class probabilities:')
for i, prob in enumerate(probabilities[0]):
    print(f'  {DISEASE_LABELS[i]:>25s}: {prob:.2%}')

## 10. Feature Importance (Random Forest)

In [ ]:
importances = rf_clf.model.feature_importances_
feature_names = X.columns
sorted_idx = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances[sorted_idx], color='#3498db')
plt.xticks(range(len(importances)), feature_names[sorted_idx], rotation=45, ha='right')
plt.ylabel('Importance')
plt.title('Feature Importance — Random Forest')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated a complete ECG analysis and ML disease prediction pipeline:

| Step | Description |
|------|-------------|
| Signal Generation | Synthetic ECG signals for 5 cardiac conditions |
| Preprocessing | Bandpass filtering, normalization, R-peak detection |
| Feature Extraction | 19 time-domain & frequency-domain features |
| Model Training | Random Forest, Gradient Boosting, SVM |
| Evaluation | Accuracy, classification report, confusion matrix |
| Prediction | End-to-end prediction with probability estimates |

### Next Steps
- Replace synthetic data with real ECG datasets (e.g., MIT-BIH, PTB-XL)
- Add deep learning models (CNN, LSTM) for raw signal classification
- Implement real-time ECG monitoring and prediction
- Add more cardiac conditions and fine-tune feature engineering